[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hentablihamza/fingerprint-similarity-guide/blob/main/notebooks/09_tversky_substructure.ipynb)

# 09 · Tversky & Substructure Search

*Fingerprint & Similarity Guide · Hamza Hentabli, Ph.D. · ~40 min*

Most similarity is symmetric: sim(A,B) = sim(B,A). But many real questions are **directional** — "which library molecules *contain* my scaffold?" The asymmetric **Tversky** coefficient answers those, and it is the right tool when a small query must match large molecules that embed it. We contrast it with Tanimoto and with exact substructure matching.

**Learning objectives.** By the end of this notebook you will be able to:

- use Tversky (α,β) to express 'query contained in target' and vice-versa
- run a scaffold-containment search that Tanimoto gets wrong
- relate fingerprint-Tversky screening to exact SMARTS substructure matching
- pick α,β for substructure vs superstructure queries

In [1]:
# Setup: on Google Colab this installs dependencies and clones the repo so the
# cached data and the shared `fsg` helper module are available. Running locally,
# install requirements.txt once and this cell just locates the files.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
OFFLINE = os.environ.get("FSG_OFFLINE", "0") == "1"   # CI sets this; no notebook needs the network
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "rdkit", "scikit-learn", "scipy", "pandas", "matplotlib"], check=True)
    if not os.path.isdir("fingerprint-similarity-guide"):
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/hentablihamza/fingerprint-similarity-guide.git"], check=True)
    ROOT = "fingerprint-similarity-guide"
else:
    ROOT = ".." if os.path.isdir("../data") else "."
DATA_DIR = os.path.join(ROOT, "data")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import fsg   # shared fingerprint factory + metrics + evaluation

print("Colab:", IN_COLAB, "| offline:", OFFLINE, "| data:", DATA_DIR)
print("Fingerprints available:", ", ".join(fsg.FP_NAMES))

Colab: False | offline: True | data: ../data
Fingerprints available: ECFP4-1024, ECFP4-2048, ECFP6-2048, ECFP4-counts, FCFP4-2048, RDKit, AtomPair, Torsion, MACCS, Avalon


In [2]:
import numpy as np, pandas as pd
from rdkit import Chem
egfr = fsg.load_mols(pd.read_csv(f"{DATA_DIR}/egfr_chembl37_ic50.csv"))
print(len(egfr), "EGFR inhibitors")

6871 EGFR inhibitors


## 1. The containment problem

The 4-anilinoquinazoline core is tiny next to a full inhibitor. Tanimoto punishes that size gap, so a molecule that *contains* the core still scores low. Tversky with **α=1, β=0** asks only "are the query's bits present in the target?" — the containment question.

In [3]:
core = Chem.MolFromSmiles("c1ccc(Nc2ncnc3ccccc23)cc1")   # 4-anilinoquinazoline
core_fp = fsg.make_fp("ECFP4-2048", core)
egfr_fps = fsg.make_fps("ECFP4-2048", list(egfr["mol"]))

tani = np.array(fsg.bulk_similarity("tanimoto", core_fp, egfr_fps))
tver = np.array(fsg.bulk_similarity("tversky", core_fp, egfr_fps, alpha=1.0, beta=0.0))
print(f"core vs library: mean Tanimoto={tani.mean():.3f}, mean Tversky(1,0)={tver.mean():.3f}")
print(f"molecules scoring >=0.9: Tanimoto={int((tani>=0.9).sum())}, Tversky(1,0)={int((tver>=0.9).sum())}")

core vs library: mean Tanimoto=0.162, mean Tversky(1,0)=0.461
molecules scoring >=0.9: Tanimoto=0, Tversky(1,0)=1


Tversky(1,0) flags many molecules as 'containing the core' at high score, while Tanimoto hides them below the size penalty. For a scaffold query, Tversky is the correct screen.

## 2. Does fingerprint-Tversky agree with exact substructure matching?

The ground truth for containment is an RDKit **substructure (SMARTS) match**. We check how well high Tversky(1,0) predicts a true substructure hit.

In [4]:
has_core = egfr["mol"].apply(lambda m: m.HasSubstructMatch(core)).to_numpy()
order = np.argsort(-tver)
topN = 200
hits_in_top = has_core[order[:topN]].sum()
print(f"True core-containing molecules in library: {int(has_core.sum())}")
print(f"Of the top {topN} by Tversky(1,0), {int(hits_in_top)} truly contain the core "
      f"({100*hits_in_top/topN:.0f}% precision)")
# contrast with Tanimoto top-N
order_t = np.argsort(-tani)
print(f"Of the top {topN} by Tanimoto, {int(has_core[order_t[:topN]].sum())} truly contain the core")

True core-containing molecules in library: 1771
Of the top 200 by Tversky(1,0), 188 truly contain the core (94% precision)
Of the top 200 by Tanimoto, 171 truly contain the core


Tversky(1,0) ranks true substructure matches far higher than Tanimoto does. Fingerprint-Tversky is a fast, approximate substructure screen — useful to pre-filter millions of molecules before the slower exact SMARTS match.

## 3. Superstructure: flip α and β

Swap to **α=0, β=1** to ask the opposite — "is the *target* contained in the query?" — the superstructure search. A large query will then score high against its own fragments.

In [5]:
big = egfr["mol"].iloc[int(np.argmax([m.GetNumAtoms() for m in egfr["mol"]]))]
big_fp = fsg.make_fp("ECFP4-2048", big)
frag = Chem.MolFromSmiles("c1ccncc1")
frag_fp = fsg.make_fp("ECFP4-2048", frag)
print("query=big molecule, target=pyridine fragment")
print("Tversky a=1,b=0 (big contained in frag?):", round(fsg.similarity("tversky", big_fp, frag_fp, 1.0, 0.0), 3))
print("Tversky a=0,b=1 (frag contained in big?):", round(fsg.similarity("tversky", big_fp, frag_fp, 0.0, 1.0), 3))

query=big molecule, target=pyridine fragment
Tversky a=1,b=0 (big contained in frag?): 0.029
Tversky a=0,b=1 (frag contained in big?): 0.333


The asymmetry is the whole point: α weights the query's unique bits, β the target's. Choose the direction that matches your question.

## Summary

- **Tversky(α=1, β=0)** = substructure/containment screen; it beats Tanimoto when a small query must match larger molecules.
- Fingerprint-Tversky is a fast **pre-filter** for exact SMARTS substructure search.
- **α=0, β=1** flips it to a superstructure query.


### ✏️ Exercise 1

How many EGFR molecules score Tversky(1,0) ≥ 0.95 to the core but Tanimoto < 0.5?

<details>
<summary><b>Show solution</b></summary>

```python
mask = (tver >= 0.95) & (tani < 0.5)
print(int(mask.sum()), "molecules hidden from Tanimoto but caught by Tversky")
```

</details>

### ✏️ Exercise 2

Compute the precision of the top-50 Tversky(1,0) hits for true core containment.

<details>
<summary><b>Show solution</b></summary>

```python
print(round(100*has_core[order[:50]].mean(), 1), "% precision in top 50")
```

</details>